# Tech Challenge - Fase 1: Análise de Exportação de Vinhos

**Autor:** Lucas Fontes Culatrelli
**Pós-Graduação:** Data Analytics (FIAP)

---

## Objetivo do Projeto
Atuando como Analista de Dados de uma grande vinícola, o objetivo deste projeto é analisar o histórico de exportações de vinhos do Brasil (dados da Embrapa) para identificar padrões, oportunidades e riscos.

A análise foca nos últimos **15 anos** e busca responder:
1. Como evoluiu nosso faturamento?
2. Quem são nossos maiores clientes?
3. Qual a estratégia para o futuro?

## 1. Configuração do Ambiente
Importação das bibliotecas essenciais para manipulação de dados (`pandas`) e visualização (`plotly`, `seaborn`).

In [ ]:
#Manipulação de dados
import pandas as pd
import numpy as np

#Visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)

## 2. Extração e Limpeza
Os dados originais (`Exportacao.csv`) apresentam um problema estrutural: os anos estão distribuídos em colunas (formato "Largo"). Além disso, o CSV mistura colunas de **Quantidade (Kg)** e **Valor (US$)** com nomes duplicados.

**Estratégia de Limpeza:**
1. Identificar colunas duplicadas (o Pandas adiciona `.1` nas colunas de Valor).
2. Utilizar a função `melt` para transformar colunas em linhas.
3. Unificar as tabelas para criar um dataset limpo e "linkável".
4. Filtrar apenas os últimos **15 anos** (2009-2023) conforme requisito do projeto.

In [ ]:
def limpar_dataset_exportacao(caminho_arquivo):
    # Carregamento do arquivo Exportacao.csv
    df = pd.read_csv(caminho_arquivo, sep=';', encoding='utf-8')

In [ ]:
# Carrega o dataset inicialmente para as etapas de limpeza demonstrativas
df = pd.read_csv('Exportacao.csv', sep=';', encoding='utf-8')

In [ ]:
# Identificar colunas de quantidade e valor
colunas = df.columns
cols_qtd = [c for c in colunas if c not in ['Id', 'País'] and '.1' not in c]
cols_val = [c for c in colunas if '.1' in c]

In [ ]:
# Criamos um DF só para quantidade
df_qtd = pd.melt(df, id_vars=['Id', 'País'], value_vars=cols_qtd,
                        var_name='Ano', value_name='Quantidade_Kg')

In [ ]:
# Criamos um DF só para valor
df_val = df.melt(id_vars=['Id', 'País'], value_vars=cols_val,
                 var_name='Ano_Temp', value_name='Valor_USD')

In [ ]:
# Limpar a coluna auxiliar de ano no df_val (remover o ".1") para fazer o join
df_val['Ano'] = df_val['Ano_Temp'].str.replace('.1', '', regex=False)
df_val = df_val.drop('Ano_Temp', axis=1)

In [ ]:
# Unir (Merge) as duas bases
df_final = pd.merge(df_qtd, df_val, on=['Id', 'País', 'Ano'])

In [ ]:
# Converter Ano para número inteiro
df_final['Ano'] = df_final['Ano'].astype(int)

In [ ]:
# Executando a função
df_full = limpar_dataset_exportacao('Exportacao.csv')

In [ ]:
df_analise = df_final[df_final['Ano'] >= 2009].copy()

In [ ]:
# país de origem
df_analise['País_Origem'] = 'Brasil'

# conversão Kg → Litros (1Kg = 1L)
df_analise['Quantidade_Litros'] = df_analise['Quantidade_Kg']


In [ ]:
# Calcular preço médio por litro
df_analise['Preco_Medio_Litro'] = np.where(
df_analise['Quantidade_Litros'] > 0,
df_analise['Valor_USD'] / df_analise['Quantidade_Litros'],
0
)

In [ ]:
# Renomear e reordenar colunas
df_analise = df_analise.rename(columns={'País': 'País_Destino'})
df_analise = df_analise[[
'Id', 'País_Origem', 'País_Destino', 'Ano',
'Quantidade_Litros', 'Valor_USD', 'Preco_Medio_Litro'
]]


In [ ]:
# Remover registros sem exportação
df_analise = df_analise[
    (df_analise['Quantidade_Litros'] > 0) | (df_analise['Valor_USD'] > 0)
].reset_index(drop=True)

In [ ]:
df_analise.to_csv('df_analise_exportacao.csv', index=False)


In [ ]:
# Visualizando para garantir que funcionou
display(df_analise.head())
print(f"Total de registros para análise: {df_analise.shape[0]}")

,Id,País_Origem,País_Destino,Ano,Quantidade_Litros,Valor_USD,Preco_Medio_Litro
0,3,Brasil,"Alemanha, República Democrática",2009,225086,393482,1.748141
1,4,Brasil,Angola,2009,54786,84235,1.537528
2,7,Brasil,Antilhas Holandesas,2009,8235,10651,1.293382
3,9,Brasil,Argentina,2009,162,4523,27.919753
4,11,Brasil,Austrália,2009,1014,9195,9.068047


Total de registros para análise: 744


##  3. Panorama Geral: Evolução das Exportações
Analisando a série temporal para entender a volatilidade do mercado e tendências de crescimento.


In [ ]:
# Agregações por ano
export_ano = df_analise.groupby('Ano').agg({
    'Valor_USD': 'sum',
    'Quantidade_Litros': 'sum',
    'País_Destino': 'nunique'
}).reset_index()

export_ano['Preco_Medio'] = export_ano['Valor_USD'] / export_ano['Quantidade_Litros']
export_ano.columns = ['Ano', 'Faturamento_USD', 'Volume_Litros',
                       'Num_Paises', 'Preco_Medio_Litro']


In [ ]:
# Calcular variações ano a ano
export_ano['Var_Faturamento_%'] = export_ano['Faturamento_USD'].pct_change() * 100
export_ano['Var_Volume_%'] = export_ano['Volume_Litros'].pct_change() * 100

In [ ]:
# Estatísticas gerais
total_faturamento = export_ano['Faturamento_USD'].sum()
total_volume = export_ano['Volume_Litros'].sum()
preco_medio_geral = total_faturamento / total_volume

print(f"\n💰 Faturamento Total (15 anos): US$ {total_faturamento:,.2f}")
print(f"📦 Volume Total (15 anos): {total_volume:,.0f} litros")
print(f"💵 Preço Médio Global: US$ {preco_medio_geral:.2f}/litro")


💰 Faturamento Total (15 anos): US$ 114,449,292.00
📦 Volume Total (15 anos): 83,174,997 litros
💵 Preço Médio Global: US$ 1.38/litro


Observamos um comportamento oscilatório. Picos de exportação geralmente estão atreladps a taxas de câmbio favoráveis (Dólar alto favorece a exportaçã) e safras recordes. Em anos como 2013 ou 2021, houve um salto significativo.

## 4. Visualização - Evolução temporal


In [ ]:
# Criar figura com subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Evolução do Faturamento (US$)',
        'Evolução do Volume (Litros)',
        'Variação Anual do Faturamento (%)',
        'Preço Médio por Litro (US$)'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# Gráfico 1: Faturamento
fig.add_trace(
    go.Scatter(
        x=export_ano['Ano'],
        y=export_ano['Faturamento_USD'],
        mode='lines+markers',
        name='Faturamento',
        line=dict(color='#2E86AB', width=3),
        marker=dict(size=8),
        hovertemplate='%{y:,.0f} USD<extra></extra>'
    ),
    row=1, col=1
)

# Gráfico 2: Volume
fig.add_trace(
    go.Scatter(
        x=export_ano['Ano'],
        y=export_ano['Volume_Litros'],
        mode='lines+markers',
        name='Volume',
        line=dict(color='#A23B72', width=3),
        marker=dict(size=8),
        hovertemplate='%{y:,.0f} L<extra></extra>'
    ),
    row=1, col=2
)

# Gráfico 3: Variação percentual
colors = ['#06A77D' if x > 0 else '#D62246' for x in export_ano['Var_Faturamento_%'].fillna(0)]
fig.add_trace(
    go.Bar(
        x=export_ano['Ano'],
        y=export_ano['Var_Faturamento_%'],
        name='Variação %',
        marker_color=colors,
        hovertemplate='%{y:.1f}%<extra></extra>'
    ),
    row=2, col=1
)

# Gráfico 4: Preço médio
fig.add_trace(
    go.Scatter(
        x=export_ano['Ano'],
        y=export_ano['Preco_Medio_Litro'],
        mode='lines+markers',
        name='Preço Médio',
        line=dict(color='#F18F01', width=3),
        marker=dict(size=8),
        hovertemplate='US$ %{y:.2f}/L<extra></extra>'
    ),
    row=2, col=2
)


# Layout
fig.update_layout(
    height=800,
    showlegend=False,
    title_text="<b>Panorama das Exportações de Vinho Brasileiro (2009-2023)</b>",
    title_x=0.5,
    title_font_size=18
)

fig.update_xaxes(title_text="Ano", row=2, col=1)
fig.update_xaxes(title_text="Ano", row=2, col=2)
fig.update_yaxes(title_text="US$", row=1, col=1)
fig.update_yaxes(title_text="Litros", row=1, col=2)
fig.update_yaxes(title_text="%", row=2, col=1)
fig.update_yaxes(title_text="US$/L", row=2, col=2)

fig.show()

## 5. ANÁLISE POR PAÍS - TOP PARCEIROS

In [ ]:
# Agregar por país
ranking_paises = df_analise.groupby('País_Destino').agg({
    'Valor_USD': 'sum',
    'Quantidade_Litros': 'sum'
}).reset_index()

ranking_paises['Preco_Medio'] = (
    ranking_paises['Valor_USD'] / ranking_paises['Quantidade_Litros']
)

In [ ]:
# Top 15 por faturamento
top_15_valor = ranking_paises.nlargest(15, 'Valor_USD')

In [ ]:
# Criar gráfico de barras horizontal
fig = go.Figure()

fig.add_trace(go.Bar(
    y=top_15_valor['País_Destino'][::-1],
    x=top_15_valor['Valor_USD'][::-1],
    orientation='h',
    marker=dict(
        color=top_15_valor['Preco_Medio'][::-1],
        colorscale='RdYlGn',
        showscale=True,
        colorbar=dict(title="Preço<br>Médio<br>US$/L")
    ),
    text=top_15_valor['Valor_USD'][::-1],
    texttemplate='US$ %{text:,.0f}',
    textposition='outside',
    hovertemplate=(
        '<b>%{y}</b><br>' +
        'Faturamento: US$ %{x:,.0f}<br>' +
        '<extra></extra>'
    )
))

fig.update_layout(
    title='<b>Top 15 Países por Faturamento (2009-2023)</b>',
    title_x=0.5,
    xaxis_title='Faturamento Total (US$)',
    yaxis_title='',
    height=600,
    showlegend=False
)

fig.show()

In [ ]:
# Análise de concentração
top_5_participacao = (top_15_valor.head(5)['Valor_USD'].sum() /
                      ranking_paises['Valor_USD'].sum() * 100)

print(f"Concentração: Top 5 países representam {top_5_participacao:.1f}% do faturamento")


Concentração: Top 5 países representam 74.2% do faturamento


## 6. ANÁLISE DETALHADA - PARAGUAI

In [ ]:
pais_alvo = "Paraguai"
df_paraguai = df_analise[df_analise['País_Destino'] == pais_alvo].groupby('Ano').agg({
    'Quantidade_Litros': 'sum',
    'Valor_USD': 'sum'
}).reset_index()

df_paraguai['Preco_Medio'] = df_paraguai['Valor_USD'] / df_paraguai['Quantidade_Litros']

# Gráfico de eixo duplo
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Volume (barras)
fig.add_trace(
    go.Bar(
        x=df_paraguai['Ano'],
        y=df_paraguai['Quantidade_Litros'],
        name="Volume",
        marker_color='rgba(46, 134, 171, 0.7)',
        hovertemplate='%{y:,.0f} litros<extra></extra>'
    ),
    secondary_y=False
)

# Faturamento (linha)
fig.add_trace(
    go.Scatter(
        x=df_paraguai['Ano'],
        y=df_paraguai['Valor_USD'],
        name="Faturamento",
        mode='lines+markers',
        line=dict(color='#D62246', width=3),
        marker=dict(size=10),
        hovertemplate='US$ %{y:,.0f}<extra></extra>'
    ),
    secondary_y=True
)

# Preço médio (linha tracejada)
fig.add_trace(
    go.Scatter(
        x=df_paraguai['Ano'],
        y=df_paraguai['Preco_Medio'],
        name="Preço Médio",
        mode='lines',
        line=dict(color='#F18F01', width=2, dash='dash'),
        hovertemplate='US$ %{y:.2f}/L<extra></extra>'
    ),
    secondary_y=True
)

fig.update_layout(
    title=f'<b>Análise Detalhada: {pais_alvo}</b>',
    title_x=0.5,
    height=500,
    hovermode='x unified'
)

fig.update_xaxes(title_text="Ano")
fig.update_yaxes(title_text="Volume (Litros)", secondary_y=False)
fig.update_yaxes(title_text="Faturamento (US$) / Preço Médio (US$/L)", secondary_y=True)

fig.show()

In [ ]:
# Estatísticas do Paraguai
print(f"ANÁLISE: {pais_alvo}")
print(f"   • Faturamento Total: US$ {df_paraguai['Valor_USD'].sum():,.2f}")
print(f"   • Volume Total: {df_paraguai['Quantidade_Litros'].sum():,.0f} litros")
print(f"   • Preço Médio: US$ {df_paraguai['Preco_Medio'].mean():.2f}/litro")
print(f"   • Crescimento Volume (2009-2023): {((df_paraguai.iloc[-1]['Quantidade_Litros'] / df_paraguai.iloc[0]['Quantidade_Litros'] - 1) * 100):.1f}%")

ANÁLISE: Paraguai
   • Faturamento Total: US$ 42,862,206.00
   • Volume Total: 30,803,247 litros
   • Preço Médio: US$ 1.37/litro
   • Crescimento Volume (2009-2023): 676.4%


## 7. SEGMENTAÇÃO DE MERCADOS

In [ ]:
#Criar segmentos baseados em volume e preço
ranking_paises['Segmento'] = pd.cut(
    ranking_paises['Preco_Medio'],
    bins=[0, 1.5, 3.0, np.inf],
    labels=['Econômico', 'Médio', 'Premium']
)

# Gráfico de dispersão
fig = px.scatter(
    ranking_paises,
    x='Quantidade_Litros',
    y='Preco_Medio',
    size='Valor_USD',
    color='Segmento',
    hover_name='País_Destino',
    labels={
        'Quantidade_Litros': 'Volume Total (Litros)',
        'Preco_Medio': 'Preço Médio (US$/Litro)',
        'Valor_USD': 'Faturamento'
    },
    title='<b>Segmentação de Mercados: Volume vs Preço Médio</b>',
    color_discrete_map={
        'Econômico': '#2E86AB',
        'Médio': '#F18F01',
        'Premium': '#06A77D'
    }
)

fig.update_layout(height=600)
fig.show()

# Análise por segmento
segmento_analise = ranking_paises.groupby('Segmento').agg({
    'País_Destino': 'count',
    'Valor_USD': 'sum',
    'Quantidade_Litros': 'sum'
}).reset_index()

segmento_analise.columns = ['Segmento', 'Num_Paises', 'Faturamento', 'Volume']
segmento_analise['Part_Faturamento_%'] = (
    segmento_analise['Faturamento'] / segmento_analise['Faturamento'].sum() * 100
)

display(segmento_analise.round(2))

,Segmento,Num_Paises,Faturamento,Volume,Part_Faturamento_%
0,Econômico,17,71550580,67617880,62.52
1,Médio,40,23600221,10746338,20.62
2,Premium,66,19298491,4810779,16.86


In [ ]:
# Criar segmentos
ranking_paises['Segmento'] = pd.cut(
    ranking_paises['Preco_Medio'],
    bins=[0, 1.5, 3.0, np.inf],
    labels=['Econômico', 'Médio', 'Premium']
)

# Filtrar apenas Premium
ranking_premium = ranking_paises.loc[
    ranking_paises['Segmento'] == 'Premium'
].copy()

ranking_premium = ranking_premium.sort_values('Valor_USD', ascending=False)

# Gráfico — apenas Premium
fig = px.scatter(
    ranking_premium,
    x='Quantidade_Litros',
    y='Preco_Medio',
    size='Valor_USD',
    hover_name='País_Destino',
    labels={
        'Quantidade_Litros': 'Volume Total (Litros)',
        'Preco_Medio': 'Preço Médio (US$/Litro)',
        'Valor_USD': 'Faturamento'
    },
    title='<b>Mercados Premium: Volume vs Preço Médio</b>'
)

fig.update_layout(height=600)
fig.show()

display(ranking_premium.head(20))



,País_Destino,Valor_USD,Quantidade_Litros,Preco_Medio,Segmento
98,Reino Unido,4640935,1150780,4.032860,Premium
92,Países Baixos,3012934,897986,3.355213,Premium
1,"Alemanha, República Democrática",2148277,648115,3.314654,Premium
19,Bélgica,1382940,399239,3.463940,Premium
23,Canadá,1059120,186081,5.691715,Premium
105,Suíça,718710,101010,7.115236,Premium
95,Polônia,553317,134483,4.114401,Premium
47,Finlândia,537443,86768,6.194023,Premium
60,Hong Kong,511459,162957,3.138613,Premium
2,Angola,505106,143147,3.528583,Premium


In [ ]:
# Criar segmentos
ranking_paises['Segmento'] = pd.cut(
    ranking_paises['Preco_Medio'],
    bins=[0, 1.5, 3.0, np.inf],
    labels=['Econômico', 'Médio', 'Premium']
)

# Filtrar apenas Premium
ranking_med= ranking_paises.loc[
    ranking_paises['Segmento'] == 'Médio'
].copy()

ranking_med = ranking_med.sort_values('Valor_USD', ascending=False)

# Gráfico — apenas Premium
fig = px.scatter(
    ranking_med,
    x='Quantidade_Litros',
    y='Preco_Medio',
    size='Valor_USD',
    hover_name='País_Destino',
    labels={
        'Quantidade_Litros': 'Volume Total (Litros)',
        'Preco_Medio': 'Preço Médio (US$/Litro)',
        'Valor_USD': 'Faturamento'
    },
    title='<b>Mercados Médios: Volume vs Preço Médio</b>'
)

fig.update_layout(height=600)
fig.show()

display(ranking_med.head(20))



,País_Destino,Valor_USD,Quantidade_Litros,Preco_Medio,Segmento
44,Estados Unidos,9309051,3349299,2.779403,Médio
27,China,4903695,2574686,1.904580,Médio
43,Espanha,3805889,1990238,1.912278,Médio
68,Japão,2257163,972341,2.321370,Médio
48,França,707581,308740,2.291835,Médio
96,Portugal,578788,384238,1.506327,Médio
37,Curaçao,361490,216254,1.671599,Médio
119,Venezuela,309340,196545,1.573889,Médio
84,Nigéria,264004,164881,1.601179,Médio
49,Gana,143512,91247,1.572786,Médio


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### . CONCLUSÕES E RECOMENDAÇÕES

In [ ]:

# MÉTRICAS

fat_paraguai = ranking_paises.loc[
    ranking_paises['País_Destino'] == 'Paraguai', 'Valor_USD'
].sum()

share_paraguai = (fat_paraguai / total_faturamento) * 100

preco_paraguai = ranking_paises.loc[
    ranking_paises['País_Destino'] == 'Paraguai', 'Preco_Medio'
].iat[0]

part_economico = segmento_analise.loc[
    segmento_analise['Segmento'] == 'Econômico', 'Part_Faturamento_%'
].iat[0]

part_medio = segmento_analise.loc[
    segmento_analise['Segmento'] == 'Médio', 'Part_Faturamento_%'
].iat[0]

part_premium = segmento_analise.loc[
    segmento_analise['Segmento'] == 'Premium', 'Part_Faturamento_%'
].iat[0]


# RELATÓRIO

conclusao = f"""
## Relatório de Exportações de Vinho Brasileiro

### Sumário Executivo

Nos últimos 15 anos (2009–2023), o Brasil exportou **USD {total_faturamento/1e6:.1f} milhões** em vinhos, totalizando **{total_volume/1e6:.1f} milhões de litros**.

A análise aponta dois pontos críticos:
- Alta dependência de mercados de baixo valor agregado
- Exposição relevante à volatilidade cambial

O preço médio global é de **USD {preco_medio_geral:.2f}/litro**, abaixo do potencial do produto brasileiro.

---

## Concentração de Risco: Paraguai

O Paraguai representa **{share_paraguai:.1f}%** do faturamento total das exportações.

**Características do mercado paraguaio:**
- Vantagens: logística eficiente, Mercosul, canal consolidado
- Desvantagens: preço médio reduzido (**USD {preco_paraguai:.2f}/L**)
- Perfil: vinho de mesa, alto volume e baixa margem

---

## Segmentação de Mercados (por preço médio)

A análise dos **{ranking_paises.shape[0]}** países consumidores de vinho brasileiro, permite separar os destinos em três grupos:

- **Segmento Econômico** (preço médio inferior a USD 1,50/L): representa **{part_economico:.1f}%** do faturamento. Em geral, é um perfil orientado a volume, com baixa captura de margem.\n
- **Segmento Médio** (entre USD 1,50 e USD 3,00/L): representa **{part_medio:.1f}%** do faturamento. É um segmento de equilíbrio, com potencial de otimização via mix e posicionamento.\n
- **Segmento Premium** (acima de USD 3,00/L): representa **{part_premium:.1f}%** do faturamento. Esses mercados remuneram múltiplos superiores por litro, sendo o principal vetor de captura de valor.

**Principal oportunidade:** apenas **{part_premium:.1f}%** do faturamento atual provém de mercados premium, que pagam significativamente mais por litro exportado.

---

## Recomendações Estratégicas

**Migrar de volume para valor, reduzindo concentração de risco e ampliando margens.**

### Curto Prazo
1. **Reduzir concentração:** diminuir participação do Paraguai no faturamento (ex.: 37% → 28%) via crescimento de outros mercados.
2. **Expandir em mercados premium:** intensificar presença em destinos de maior preço médio, com parcerias comerciais e importadores especializados.
3. **Melhorar precificação:** implementar pricing diferenciado por mercado e portfólio, elevando o preço médio global (ex.: USD 1,38 → USD 1,65/L).

### Médio Prazo
- **Construir marcas de maior valor agregado:** identidade e posicionamento, com certificações de origem e comunicação consistente.
- **Penetração seletiva:** expandir em novos mercados de maior valor médio (ex.: Estados Unidos, China, Reino Unido, Países Baixos e Alemanha) com abordagem piloto antes de escalar.
Deve ser levado em consideração desvantagens logísticas em relação mercados europeus que são fortemente atendidos por produtores locais.

### Longo Prazo
- **Consolidar posição premium:** meta de 35–40% do faturamento no segmento premium, elevando o preço médio global para USD 2,20–2,50/L.
- **Inovação:** lançamento de linhas super-premium e storytelling robusto (sustentabilidade, origem, diferenciais de qualidade do produto).


## Conclusão

A indústria brasileira de vinhos está diante de uma decisão estratégica:
continuar competindo por volume em mercados de baixo valor ou reposicionar-se como fornecedor de vinhos premium.

**Recomendação:** aprovar o plano de desconcentração e migração para mercados premium.

"""

display(Markdown(conclusao))




## Relatório de Exportações de Vinho Brasileiro

### Sumário Executivo

Nos últimos 15 anos (2009–2023), o Brasil exportou **USD 114.4 milhões** em vinhos, totalizando **83.2 milhões de litros**.

A análise aponta dois pontos críticos:
- Alta dependência de mercados de baixo valor agregado  
- Exposição relevante à volatilidade cambial  

O preço médio global é de **USD 1.38/litro**, abaixo do potencial do produto brasileiro.

---

## Concentração de Risco: Paraguai

O Paraguai representa **37.5%** do faturamento total das exportações.

**Características do mercado paraguaio:**
- Vantagens: logística eficiente, Mercosul, canal consolidado  
- Desvantagens: preço médio reduzido (**USD 1.39/L**)  
- Perfil: vinho de mesa, alto volume e baixa margem  

---

## Segmentação de Mercados (por preço médio)

A análise dos **123** países consumidores de vinho brasileiro, permite separar os destinos em três grupos:

- **Segmento Econômico** (preço médio inferior a USD 1,50/L): representa **62.5%** do faturamento. Em geral, é um perfil orientado a volume, com baixa captura de margem.

- **Segmento Médio** (entre USD 1,50 e USD 3,00/L): representa **20.6%** do faturamento. É um segmento de equilíbrio, com potencial de otimização via mix e posicionamento.

- **Segmento Premium** (acima de USD 3,00/L): representa **16.9%** do faturamento. Esses mercados remuneram múltiplos superiores por litro, sendo o principal vetor de captura de valor.

**Principal oportunidade:** apenas **16.9%** do faturamento atual provém de mercados premium, que pagam significativamente mais por litro exportado.

---

## Recomendações Estratégicas

**Migrar de volume para valor, reduzindo concentração de risco e ampliando margens.**

### Curto Prazo
1. **Reduzir concentração:** diminuir participação do Paraguai no faturamento (ex.: 37% → 28%) via crescimento de outros mercados.
2. **Expandir em mercados premium:** intensificar presença em destinos de maior preço médio, com parcerias comerciais e importadores especializados.
3. **Melhorar precificação:** implementar pricing diferenciado por mercado e portfólio, elevando o preço médio global (ex.: USD 1,38 → USD 1,65/L).

### Médio Prazo
- **Construir marcas de maior valor agregado:** identidade e posicionamento, com certificações de origem e comunicação consistente.
- **Penetração seletiva:** expandir em novos mercados de maior valor médio (ex.: Estados Unidos, China, Reino Unido, Países Baixos e Alemanha) com abordagem piloto antes de escalar.
Deve ser levado em consideração desvantagens logísticas em relação mercados europeus que são fortemente atendidos por produtores locais.

### Longo Prazo
- **Consolidar posição premium:** meta de 35–40% do faturamento no segmento premium, elevando o preço médio global para USD 2,20–2,50/L.
- **Inovação:** lançamento de linhas super-premium e storytelling robusto (sustentabilidade, origem, diferenciais de qualidade do produto).


## Conclusão

A indústria brasileira de vinhos está diante de uma decisão estratégica:  
continuar competindo por volume em mercados de baixo valor ou reposicionar-se como fornecedor de vinhos premium.

**Recomendação:** aprovar o plano de desconcentração e migração para mercados premium.

